In [19]:
import asyncio
import csv
import json
import os
import random
import sys
import time
from datetime import datetime
from pathlib import Path

import getpass
import pandas as pd
from playwright.sync_api import sync_playwright
from playwright_stealth import Stealth


In [20]:
# ============================================================
# KONFIGURASI
# ============================================================

# ------------------------------------------------------------
# FILE EXCEL INPUT
# ------------------------------------------------------------
EXCEL_PATH = Path("../Tabel R.133. Kombinasi kepemilikan 2+ aset dengan kode 2 (sisa bug prelist sebelum 8 Juli 2026) (MIKRO).xlsx").resolve()   # <-- ganti dengan nama file Excel Anda
SHEET_NAME = 0  # index sheet atau nama sheet
HEADER_ROW = 1 
KOLOM_ID = "assignment_id"                        # nama kolom di Excel


# ------------------------------------------------------------
# URL & LOGIN
# ------------------------------------------------------------
LINK_BASE_URL = "https://fasih-dashboard.bps.go.id/"
LINK_SQLLAB_URL = "https://fasih-dashboard.bps.go.id/superset/sqllab/"
URL_SQL_EXECUTE = "https://fasih-dashboard.bps.go.id/api/v1/sqllab/execute/"

SQLLAB_USERNAME = os.environ.get("SQLLAB_USERNAME")
SQLLAB_PASSWORD = os.environ.get("SQLLAB_PASSWORD")
SELECTOR_USERNAME = "xpath=//*[@id='username']"
SELECTOR_PASSWORD = "xpath=//*[@id='password']"
SELECTOR_TOMBOL_LOGIN = "xpath=//*[@id='container']/div[2]/form/button"


# ------------------------------------------------------------
# SQL LAB
# ------------------------------------------------------------
DATABASE_ID = 25
SQL_EDITOR_ID = "1036293"
SCHEMA = "tgr_fd68e454"
TAB_NAME = "Tagging level_6_full_code"

# Jumlah assignment_id per query (= batas max row hasil query)
CHUNK_SIZE = 1000
QUERY_LIMIT = 9000

# Kolom output (urutan sesuai SELECT)
KOLOM_OUTPUT = ["assignment_id", "level_6_full_code"]

MAX_RETRY = 5
REQUEST_TIMEOUT = 180000  # ms (SQL besar, timeout dibuat lebih longgar)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
OUTPUT_DIR = Path("hasil_scraping").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# BACA ASSIGNMENT_ID DARI EXCEL
# ============================================================

def baca_assignment_id():
    """
    Membaca kolom assignment_id dari Excel.
    - dibaca sebagai teks (tidak diubah jadi angka)
    - dibersihkan spasi, kosong dibuang
    - duplikat dibuang (urutan asli dipertahankan)
    """
    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW, dtype=str)

    kolom = {str(c).strip().lower(): c for c in df.columns}
    if KOLOM_ID.lower() not in kolom:
        raise KeyError(
            f"Kolom '{KOLOM_ID}' tidak ditemukan. Kolom tersedia: {list(df.columns)}"
        )

    seri = df[kolom[KOLOM_ID.lower()]].dropna().astype(str).str.strip()
    seri = seri[seri != ""]

    total_awal = len(seri)
    ids = list(dict.fromkeys(seri.tolist()))

    print("=" * 70)
    print("INPUT EXCEL")
    print("=" * 70)
    print(f"File           : {EXCEL_PATH}")
    print(f"Baris terisi   : {total_awal:,}")
    print(f"ID unik        : {len(ids):,}")
    print(f"Duplikat dibuang: {total_awal - len(ids):,}")
    return ids


def pecah_chunk(ids, ukuran=CHUNK_SIZE):
    return [ids[i:i + ukuran] for i in range(0, len(ids), ukuran)]


ASSIGNMENT_IDS = baca_assignment_id()
CHUNKS = pecah_chunk(ASSIGNMENT_IDS)

print(f"Jumlah query   : {len(CHUNKS):,} (maks {CHUNK_SIZE:,} ID per query)")

INPUT EXCEL
File           : D:\2026\Sensus Ekonomi\Tabel R.133. Kombinasi kepemilikan 2+ aset dengan kode 2 (sisa bug prelist sebelum 8 Juli 2026) (MIKRO).xlsx
Baris terisi   : 54,378
ID unik        : 54,377
Duplikat dibuang: 1
Jumlah query   : 55 (maks 1,000 ID per query)


In [15]:
# ============================================================
# MEMBUAT SQL (MINIFIED, 1 BARIS)
# ============================================================

def buat_sql(chunk):
    """
    Membuat SQL satu baris (tanpa newline) supaya aman dari batas 1000 baris.

    SELECT assignment_id, level_6_full_code FROM root_table
    WHERE assignment_id IN ('a','b','c',...)
    """
    daftar = ",".join("'" + i.replace("'", "''") + "'" for i in chunk)

    sql = (
        "SELECT assignment_id, level_6_full_code "
        "FROM root_table "
        f"WHERE assignment_id IN ({daftar})"
    )

    assert "\n" not in sql, "SQL tidak boleh mengandung baris baru"
    return sql


# Contoh SQL (3 ID pertama saja, untuk pengecekan bentuk)
print(buat_sql(ASSIGNMENT_IDS[:3]))


SELECT assignment_id, level_6_full_code FROM root_table WHERE assignment_id IN ('11c162e7-b410-4287-aeac-7714b2b1a15d','604f044a-2fd0-4af3-9287-2698c2dcb222','6692bec3-3bf3-4a34-bd1c-e91392969e00')


In [ ]:
# ============================================================
# GENERATE SQL SAJA (UNTUK DI-RUN MANUAL DI SQL LAB)
# Tidak butuh login / browser. Cukup jalankan cell konfigurasi
# dan cell buat_sql di atas, lalu cell ini.
# ============================================================

CHUNK_SIZE_MANUAL = 500     # jumlah assignment_id per query; kecilkan lagi jika masih terlalu besar
NOMOR_DICETAK = 1           # nomor query yang dicetak ke layar (mulai dari 1)

SQL_DIR = OUTPUT_DIR / "sql_manual"
SQL_DIR.mkdir(parents=True, exist_ok=True)

# Bersihkan file lama agar tidak tercampur jika CHUNK_SIZE_MANUAL diganti
for f in SQL_DIR.glob("query_*.sql"):
    f.unlink()

chunks_manual = pecah_chunk(ASSIGNMENT_IDS, CHUNK_SIZE_MANUAL)
TOTAL_QUERY = len(chunks_manual)

for n, chunk in enumerate(chunks_manual, start=1):
    (SQL_DIR / f"query_{n:03d}.sql").write_text(buat_sql(chunk), encoding="utf-8")


def tampil_sql(n):
    """Cetak query ke-n (mulai dari 1) supaya bisa di-copy ke SQL Lab."""
    if not 1 <= n <= TOTAL_QUERY:
        print(f"Nomor harus 1 sampai {TOTAL_QUERY}")
        return
    chunk = chunks_manual[n - 1]
    sql = buat_sql(chunk)
    print("=" * 70)
    print(f"QUERY {n}/{TOTAL_QUERY} | {len(chunk):,} ID | {len(sql):,} karakter")
    print("=" * 70)
    print(sql)
    print("=" * 70)


print(f"Total ID     : {len(ASSIGNMENT_IDS):,}")
print(f"Jumlah query : {TOTAL_QUERY:,} (maks {CHUNK_SIZE_MANUAL:,} ID per query)")
print(f"Semua SQL tersimpan di: {SQL_DIR}")
print("Cetak query lain dengan: tampil_sql(2), tampil_sql(3), dst.\n")

tampil_sql(NOMOR_DICETAK)


In [ ]:
# ============================================================
# FIX EVENT LOOP UNTUK PLAYWRIGHT DI JUPYTER (WINDOWS)
# ============================================================

class GunakanProactorEventLoop:
    """
    Memaksa Windows memakai ProactorEventLoop selama Playwright aktif,
    lalu mengembalikan policy semula setelah selesai.
    """

    def __enter__(self):
        self.policy_lama = None
        if sys.platform == "win32":
            self.policy_lama = asyncio.get_event_loop_policy()
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        return self

    def __exit__(self, exc_type, exc_value, tb):
        if self.policy_lama is not None:
            asyncio.set_event_loop_policy(self.policy_lama)


# ============================================================
# LOGIN
# ============================================================

def isi_username_password(page):
    page.wait_for_selector(SELECTOR_USERNAME, state="visible", timeout=15000)
    # username = input("Username SQL Lab: ").strip()
    # password = getpass.getpass("Password SQL Lab: ").strip()
    page.fill(SELECTOR_USERNAME, SQLLAB_USERNAME)
    page.fill(SELECTOR_PASSWORD, SQLLAB_PASSWORD)
    page.press(SELECTOR_PASSWORD, "Enter")
    print("Username & password terisi, form disubmit.")


def klik_tombol_login(page, max_percobaan=5, timeout_ms=15000):
    try:
        page.wait_for_selector(
            SELECTOR_TOMBOL_LOGIN, timeout=timeout_ms, state="visible"
        )
    except Exception:
        print("Tombol login tidak ditemukan.")
        print("Kemungkinan browser sudah login atau butuh autentikasi tambahan.")
        return False

    url_sebelum = page.url
    for percobaan in range(1, max_percobaan + 1):
        try:
            time.sleep(random.uniform(1, 3))
            page.locator(SELECTOR_TOMBOL_LOGIN).click(timeout=5000)
        except Exception as e:
            print(f"Percobaan {percobaan} - klik login gagal: {e}")
            page.wait_for_timeout(1500)
            continue

        page.wait_for_timeout(2000)
        ada_perubahan_url = page.url != url_sebelum
        ada_form_username = page.locator(SELECTOR_USERNAME).count() > 0

        if ada_perubahan_url or ada_form_username:
            print(f"Tombol login berhasil diproses (percobaan {percobaan}).")
            if ada_form_username:
                isi_username_password(page)
            return True
        print(f"Percobaan {percobaan} belum ada perubahan, mencoba lagi...")

    return False


def login_dashboard(page):
    print("\n" + "=" * 70)
    print("LOGIN FASIH DASHBOARD")
    print("=" * 70)
    page.goto(LINK_BASE_URL, wait_until="domcontentloaded")
    login_selesai = klik_tombol_login(page)
    if not login_selesai:
        print("\nSilakan selesaikan login/captcha jika diperlukan.")
        input("Setelah login berhasil, tekan ENTER untuk melanjutkan...")
        if page.locator(SELECTOR_USERNAME).count() > 0:
            isi_username_password(page)
    page.wait_for_timeout(3000)
    print("\nLogin selesai.")
    print("URL:", page.url)


# ============================================================
# MENANGKAP CSRF DARI REQUEST SQL LAB
# ============================================================

def pasang_penangkap_csrf(page):
    request_info = {"csrf_token": None}

    def handle_request(request):
        if "/api/v1/sqllab/execute/" not in request.url:
            return
        csrf_token = request.headers.get("x-csrftoken")
        if not csrf_token:
            return
        request_info["csrf_token"] = csrf_token
        print("\n[OK] CSRF token berhasil ditangkap.")

    page.on("request", handle_request)
    return request_info


def tunggu_csrf(page, request_info, timeout_detik=300):
    print("\n" + "=" * 70)
    print("MENUNGGU REQUEST SQL LAB")
    print("=" * 70)
    print(
        "SQL Lab sudah dibuka.\n"
        "Silakan jalankan SATU query manual (tombol RUN), contoh: SELECT 1\n"
        "Token akan ditangkap otomatis. Jangan tampilkan/kirim nilai token."
    )

    batas_waktu = time.time() + timeout_detik
    while time.time() < batas_waktu:
        if request_info["csrf_token"]:
            print("\n[OK] CSRF token sudah tersedia.")
            return request_info["csrf_token"]
        page.wait_for_timeout(500)

    raise Exception(
        "Timeout menunggu request SQL Lab. "
        "Pastikan tombol RUN sudah ditekan dan query berhasil dijalankan."
    )


# ============================================================
# PAYLOAD & EKSEKUSI SQL
# ============================================================

def buat_payload(sql):
    return {
        "database_id": DATABASE_ID,
        "json": True,
        "runAsync": False,
        "schema": SCHEMA,
        "sql": sql,
        "sql_editor_id": SQL_EDITOR_ID,
        "tab": TAB_NAME,
        "tmp_table_name": "",
        "select_as_cta": False,
        "ctas_method": "TABLE",
        "queryLimit": QUERY_LIMIT,
        "expand_data": True,
    }


def execute_query(context, sql, csrf_token, label):
    """
    Menjalankan SQL memakai session browser yang sudah login.
    Return: dict hasil | "REFRESH_CSRF" | None
    """
    payload = buat_payload(sql)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "X-CSRFToken": csrf_token,
        "Referer": LINK_SQLLAB_URL,
    }

    for percobaan in range(1, MAX_RETRY + 1):
        print("\n" + "-" * 70)
        print(f"{label} | Percobaan {percobaan}/{MAX_RETRY}")
        try:
            response = context.request.post(
                URL_SQL_EXECUTE, data=payload, headers=headers, timeout=REQUEST_TIMEOUT
            )
            print("HTTP Status:", response.status)
            response_text = response.text()

            if response.status == 200:
                try:
                    result = response.json()
                    print("[OK] Query berhasil.")
                    return result
                except Exception as e:
                    print("Response 200 tetapi bukan JSON:", e)
                    print(response_text[:500])
                    return None

            if response.status == 400:
                print(response_text[:500])
                if "csrf" in response_text.lower():
                    print("[WARNING] CSRF token tidak valid.")
                    return "REFRESH_CSRF"

            elif response.status in (401, 403):
                print("Session kemungkinan sudah expired.")
                print(response_text[:500])
                return None

            else:
                print("Request gagal.")
                print(response_text[:500])

        except Exception as e:
            print(f"Request error: {e}")

        if percobaan < MAX_RETRY:
            waktu_tunggu = min(5 * percobaan, 30)
            print(f"Menunggu {waktu_tunggu} detik...")
            time.sleep(waktu_tunggu)

    print(f"\n[GAGAL] {label}")
    return None


def ambil_rows(result):
    """Mengambil list row dari beberapa kemungkinan struktur response Superset."""
    if not isinstance(result, dict):
        return []

    data = result.get("data")
    if isinstance(data, list):
        return data

    result_data = result.get("result")
    if isinstance(result_data, list):
        for item in result_data:
            if isinstance(item, dict) and isinstance(item.get("data"), list):
                return item["data"]
    if isinstance(result_data, dict) and isinstance(result_data.get("data"), list):
        return result_data["data"]

    return []

In [24]:
# ============================================================
# STATE RESUME (PER CHUNK)
# ============================================================

def buat_state_baru(nama_file):
    return {
        "nama_file": str(nama_file),
        "excel": str(EXCEL_PATH),
        "total_id": len(ASSIGNMENT_IDS),
        "chunk_size": CHUNK_SIZE,
        "total_chunk": len(CHUNKS),
        "next_chunk": 0,          # index chunk berikutnya yang akan dijalankan
        "total_data": 0,
        "header_sudah_ditulis": False,
        "file_size": 0,
        "selesai": False,
        "updated_at": datetime.now().isoformat(),
    }


def baca_state(state_file):
    if not state_file.exists():
        return None
    try:
        with open(state_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARNING] State tidak dapat dibaca: {e}")
        return None


def simpan_state(state_file, state):
    """Simpan atomic: tulis ke file temp lalu replace."""
    temp_file = state_file.with_suffix(".tmp")
    state["updated_at"] = datetime.now().isoformat()
    with open(temp_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp_file, state_file)


def state_cocok_dengan_input(state):
    """State hanya dipakai ulang jika Excel & pengaturan chunk sama."""
    return (
        state.get("excel") == str(EXCEL_PATH)
        and state.get("total_id") == len(ASSIGNMENT_IDS)
        and state.get("chunk_size") == CHUNK_SIZE
    )


def siapkan_output():
    """
    Cari state sebelumnya yang cocok dengan input saat ini (resume),
    kalau tidak ada buat scraping baru.
    """
    state_files = sorted(
        OUTPUT_DIR.glob("*.state.json"), key=lambda x: x.stat().st_mtime, reverse=True
    )

    for state_file in state_files:
        state = baca_state(state_file)
        if not state or not state_cocok_dengan_input(state):
            continue

        nama_file = Path(state["nama_file"])
        if not nama_file.is_absolute():
            nama_file = OUTPUT_DIR / nama_file

        print("\n" + "=" * 70)
        print("SELESAI SEBELUMNYA" if state.get("selesai") else "RESUME SCRAPING")
        print("=" * 70)
        print("File CSV   :", nama_file)
        print(f"Chunk      : {state['next_chunk']:,} / {state['total_chunk']:,} selesai")
        print(f"Total data : {state['total_data']:,}")
        return nama_file, state_file, state

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    nama_file = OUTPUT_DIR / f"hasil_level6_{timestamp}.csv"
    state_file = OUTPUT_DIR / f"hasil_level6_{timestamp}.state.json"
    state = buat_state_baru(nama_file)
    simpan_state(state_file, state)

    print("\n" + "=" * 70)
    print("SCRAPING BARU")
    print("=" * 70)
    print("File CSV :", nama_file)
    print("State    :", state_file)
    return nama_file, state_file, state


def pulihkan_file_csv(nama_file, state):
    """Potong CSV ke ukuran di state agar tidak ada batch ganda setelah crash."""
    file_size = state.get("file_size", 0)
    if not nama_file.exists():
        return
    if nama_file.stat().st_size > file_size:
        print("\n[RESUME] Memulihkan ukuran CSV...")
        with open(nama_file, "r+b") as f:
            f.truncate(file_size)
        print("[OK] CSV berhasil dipulihkan.")


def simpan_batch_csv(rows, nama_file, header_sudah_ditulis):
    """Tulis satu batch row ke CSV. Return (header_sudah_ditulis, file_size)."""
    if not rows:
        return header_sudah_ditulis, (nama_file.stat().st_size if nama_file.exists() else 0)

    mode = "a" if header_sudah_ditulis else "w"

    with open(nama_file, mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f, fieldnames=KOLOM_OUTPUT, extrasaction="ignore", restval=""
        )
        if not header_sudah_ditulis:
            writer.writeheader()
        for row in rows:
            if isinstance(row, dict):
                writer.writerow(row)
        f.flush()
        os.fsync(f.fileno())

    return True, nama_file.stat().st_size


In [25]:
# ============================================================
# SCRAP SEMUA CHUNK
# ============================================================

def scrap_semua_data(context, csrf_token, nama_file, state_file, state):
    pulihkan_file_csv(nama_file, state)

    total_chunk = len(CHUNKS)
    mulai = state.get("next_chunk", 0)
    total_data = state.get("total_data", 0)
    header_sudah_ditulis = state.get("header_sudah_ditulis", False)

    print("\n" + "=" * 70)
    print("MEMULAI SCRAPING PER CHUNK ASSIGNMENT_ID")
    print("=" * 70)
    print(f"Mulai dari chunk : {mulai + 1:,} / {total_chunk:,}")
    print(f"Total data awal  : {total_data:,}")
    print("=" * 70)

    for idx in range(mulai, total_chunk):
        chunk = CHUNKS[idx]
        label = f"CHUNK {idx + 1:,}/{total_chunk:,} | {len(chunk):,} ID"

        result = execute_query(
            context=context, sql=buat_sql(chunk), csrf_token=csrf_token, label=label
        )

        if result == "REFRESH_CSRF" or result is None:
            print(f"\n[ERROR] {label} gagal. Scraping dihentikan.")
            print("Jalankan notebook lagi untuk melanjutkan dari chunk ini.")
            return nama_file, total_data, False

        rows = ambil_rows(result)
        jumlah = len(rows)
        print(f"Row diterima : {jumlah:,} (ID dikirim: {len(chunk):,})")

        if jumlah == QUERY_LIMIT and len(chunk) == QUERY_LIMIT:
            print("[INFO] Row tepat di batas limit; cek jika ada ID dengan banyak baris di root_table.")
        if jumlah < len(chunk):
            print(f"[INFO] {len(chunk) - jumlah:,} ID tidak ditemukan di root_table.")

        header_sudah_ditulis, file_size = simpan_batch_csv(
            rows=rows, nama_file=nama_file, header_sudah_ditulis=header_sudah_ditulis
        )
        total_data += jumlah

        # State disimpan SETELAH data benar-benar masuk ke disk
        state["next_chunk"] = idx + 1
        state["total_data"] = total_data
        state["header_sudah_ditulis"] = header_sudah_ditulis
        state["file_size"] = file_size
        state["selesai"] = (idx + 1 >= total_chunk)
        simpan_state(state_file, state)

        print(f"Total data   : {total_data:,}")
        print(f"Ukuran CSV   : {file_size / (1024 * 1024):,.2f} MB")

        if idx + 1 < total_chunk:
            jeda = random.uniform(1, 3)
            print(f"Menunggu {jeda:.1f} detik...")
            time.sleep(jeda)

    return nama_file, total_data, True


# ============================================================
# MAIN SCRAPING
# ============================================================

def jalankan_scraping():
    nama_file, state_file, state = siapkan_output()

    if state.get("selesai"):
        print("\nScraping sebelumnya sudah selesai.")
        print("Hapus file *.state.json di folder hasil_scraping untuk mengulang dari awal.")
        return nama_file, state.get("total_data", 0)

    with GunakanProactorEventLoop():
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=False, args=["--disable-blink-features=AutomationControlled"]
            )
            context = browser.new_context(
                accept_downloads=True,
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/126.0.0.0 Safari/537.36"
                ),
                viewport={"width": 1366, "height": 768},
                locale="id-ID",
                timezone_id="Asia/Jakarta",
            )
            page = context.new_page()
            Stealth().apply_stealth_sync(page)

            try:
                login_dashboard(page)
                request_info = pasang_penangkap_csrf(page)

                print("\nMembuka SQL Lab...")
                page.goto(LINK_SQLLAB_URL, wait_until="domcontentloaded")
                page.wait_for_timeout(5000)

                csrf_token = tunggu_csrf(page, request_info)

                print("\nMemulai scraping otomatis...")
                nama_file, total_data, selesai = scrap_semua_data(
                    context=context,
                    csrf_token=csrf_token,
                    nama_file=nama_file,
                    state_file=state_file,
                    state=state,
                )

                print("\n" + "=" * 70)
                print("SCRAPING SELESAI" if selesai else "SCRAPING BERHENTI / BELUM SELESAI")
                print("=" * 70)
                print(f"Total data : {total_data:,}")
                print(f"File CSV   : {nama_file.resolve()}")
                return nama_file, total_data

            finally:
                print("\nMenutup browser...")
                context.close()
                browser.close()


In [26]:
# ============================================================
# MAIN
# ============================================================

print("\n" + "=" * 70)
print("BOT SQL LAB FASIH - AMBIL level_6_full_code DARI assignment_id")
print("=" * 70)

nama_file, total_data = await asyncio.to_thread(jalankan_scraping)

print("\n" + "=" * 70)
print("SELESAI")
print("=" * 70)
print(f"Total data : {total_data:,}")
print(f"File CSV   : {Path(nama_file).resolve()}")



BOT SQL LAB FASIH - AMBIL level_6_full_code DARI assignment_id

RESUME SCRAPING
File CSV   : D:\2026\Sensus Ekonomi\Bot Scraper\hasil_scraping\hasil_level6_20261007_142349.csv
Chunk      : 0 / 55 selesai
Total data : 0

LOGIN FASIH DASHBOARD
Tombol login berhasil diproses (percobaan 1).
Username & password terisi, form disubmit.

Login selesai.
URL: https://fasih-dashboard.bps.go.id/superset/welcome/

Membuka SQL Lab...

MENUNGGU REQUEST SQL LAB
SQL Lab sudah dibuka.
Silakan jalankan SATU query manual (tombol RUN), contoh: SELECT 1
Token akan ditangkap otomatis. Jangan tampilkan/kirim nilai token.

[OK] CSRF token sudah tersedia.

Memulai scraping otomatis...

MEMULAI SCRAPING PER CHUNK ASSIGNMENT_ID
Mulai dari chunk : 1 / 55
Total data awal  : 0

----------------------------------------------------------------------
CHUNK 1/55 | 1,000 ID | Percobaan 1/5
HTTP Status: 500
Request gagal.
{"errors": [{"message": "SQL Syntax is too large", "error_type": "SQLLAB_TIMEOUT_ERROR", "level": "e

In [ ]:
# ============================================================
# (OPSIONAL) GABUNGKAN KE EXCEL ASLI + LAPORAN ID TIDAK DITEMUKAN
# Jalankan setelah scraping selesai.
# ============================================================

df_excel = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, dtype=str)
kolom_id_excel = next(c for c in df_excel.columns if str(c).strip().lower() == KOLOM_ID.lower())

df_hasil = pd.read_csv(nama_file, dtype=str, encoding="utf-8-sig")

# Jika satu assignment_id punya beberapa baris di root_table, ambil yang pertama
# agar jumlah baris Excel tidak bertambah.
df_hasil_unik = df_hasil.drop_duplicates(subset="assignment_id", keep="first")
if len(df_hasil_unik) != len(df_hasil):
    print(f"[INFO] {len(df_hasil) - len(df_hasil_unik):,} baris duplikat assignment_id di hasil (diambil yang pertama).")

df_gabung = df_excel.merge(
    df_hasil_unik,
    how="left",
    left_on=kolom_id_excel,
    right_on="assignment_id",
    suffixes=("", "_sqllab"),
)

tidak_ketemu = df_gabung["level_6_full_code"].isna().sum()
print(f"Baris Excel          : {len(df_excel):,}")
print(f"Berhasil dapat kode  : {len(df_gabung) - tidak_ketemu:,}")
print(f"Tidak ditemukan      : {tidak_ketemu:,}")

file_gabung = OUTPUT_DIR / f"{EXCEL_PATH.stem}_dengan_level6.xlsx"
df_gabung.to_excel(file_gabung, index=False)
print("Disimpan ke:", file_gabung)
